In [2]:
%%capture

import warnings
# suppress user warnings during execution
warnings.filterwarnings(action='ignore', category=UserWarning)

# load required dependencies
%pip install --upgrade pip
%pip install srsly

In [ ]:
# Gotriple abstracts extracted in rematch2 NER process, periods and yearspans identified.
# Now attempt to enrich the data with start-end dates for each located span.
import os
import srsly
from yearspanmatcher import YearSpan, YearSpanMatcher

for language in ["en", "fr", "es"]:
    # get the full path for the input data file
    input_directory = "./data/gotriple"
    input_file_name = f"ner-output-gotriple-abstracts-{language}.jsonl.gz"
    output_file_name = f"ner-output-gotriple-abstracts-{language}-withdates.jsonl.gz"
    input_file_path = os.path.join(input_directory, input_file_name)
    output_file_path = os.path.join(input_directory, output_file_name)

    # use specific periodo_authority_id according to language
    periodo_authority_id = ""
    if language == "es":
        # using 'SIA+ Chrono-Cultural Categories' authority
        periodo_authority_id="p07h9k6"
    elif language == "fr":
        # using 'PACTOLS chronology periods used in DOLIA data' authority
        periodo_authority_id="p02chr4"
    else:
        # using 'HE Cultural Periods' authority
        periodo_authority_id="p0kh9ds"

    # set up the matcher
    print(f"language='{language}', periodo_autority_id='{periodo_authority_id}'")            
    matcher = YearSpanMatcher(language=language, periodo_authority_id=periodo_authority_id)
    
    # attempt to get start/end year for each record in input data
    #print(f"processing {len(data)} records")
    print(f"processing data from '{input_file_path}'")
    results = []
    for record in srsly.read_gzip_jsonl(input_file_path, True):

        for item in record.get("spans", []):
            identifier = record.get("id", "")
            text = item.get("text", "")
            result = matcher.match(text)
            if (result is not None):
                item["minYear"] = YearSpan.yearToISO8601(result.minYear)
                item["maxYear"] = YearSpan.yearToISO8601(result.maxYear)
                item["isoSpan"] = result.toISO8601()
                item["duration"] = result.duration()
        results.append(record)                   
    
    print(f"writing results to {output_file_path}")
    srsly.write_gzip_jsonl(output_file_path, results)    
    print("done")  

language='en', periodo_autority_id='p0kh9ds'
processing data from './data/gotriple/ner-output-gotriple-abstracts-en.jsonl.gz'
writing results to ./data/gotriple/ner-output-gotriple-abstracts-en-withdates.jsonl.gz
done
language='fr', periodo_autority_id='p02chr4'
processing data from './data/gotriple/ner-output-gotriple-abstracts-fr.jsonl.gz'
writing results to ./data/gotriple/ner-output-gotriple-abstracts-fr-withdates.jsonl.gz
done
language='es', periodo_autority_id='p07h9k6'
processing data from './data/gotriple/ner-output-gotriple-abstracts-es.jsonl.gz'
writing results to ./data/gotriple/ner-output-gotriple-abstracts-es-withdates.jsonl.gz
done
